In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ==========================
# SETTINGS
# ==========================
initial_capital = 100000
cash = initial_capital

interval = "5m"              # intraday timeframe
period = "5d"               # last 5 days intraday data

stop_loss_pct = 0.01        # 1% SL (tight for intraday)
target_pct = 0.015          # 1.5% target
max_positions = 5

# Use only liquid stocks (IMPORTANT)
tickers = [
    "RELIANCE.NS","HDFCBANK.NS","ICICIBANK.NS",
    "INFY.NS","TCS.NS","SBIN.NS","AXISBANK.NS"
]

trade_log = []
open_positions = {}

# ==========================
# UT BOT FUNCTION
# ==========================
def compute_utbot(df, atr_period=10, multiplier=1):

    df = df.copy()

    df["tr"] = np.maximum(
        df["High"] - df["Low"],
        np.maximum(
            abs(df["High"] - df["Close"].shift()),
            abs(df["Low"] - df["Close"].shift())
        )
    )

    df["atr"] = df["tr"].rolling(atr_period).mean()

    df["upper"] = df["Close"] - multiplier * df["atr"]
    df["lower"] = df["Close"] + multiplier * df["atr"]

    trend = [1]

    for i in range(1, len(df)):
        if df["Close"].iloc[i] > df["lower"].iloc[i-1]:
            trend.append(1)
        elif df["Close"].iloc[i] < df["upper"].iloc[i-1]:
            trend.append(-1)
        else:
            trend.append(trend[-1])

    df["trend"] = trend

    df["buy"] = (df["trend"] == 1) & (df["trend"].shift() == -1)
    df["sell"] = (df["trend"] == -1) & (df["trend"].shift() == 1)

    return df

# ==========================
# LOAD DATA (FAST)
# ==========================
all_data = {}

for ticker in tickers:
    print("Loading:", ticker)

    df = yf.download(
        ticker,
        interval=interval,
        period=period,
        progress=False
    )

    if df.empty:
        continue

    df = compute_utbot(df)
    all_data[ticker] = df

# ==========================
# MASTER TIMELINE
# ==========================
all_times = sorted(
    set(time for df in all_data.values() for time in df.index)
)

equity_curve = []

# ==========================
# BACKTEST
# ==========================
for current_time in all_times:

    # ---------- SELL ----------
    for ticker in list(open_positions.keys()):

        df = all_data[ticker]

        if current_time not in df.index:
            continue

        row = df.loc[current_time]
        pos = open_positions[ticker]

        stop_price = pos["entry_price"] * (1 - stop_loss_pct)
        target_price = pos["entry_price"] * (1 + target_pct)

        if row["Close"] <= stop_price:
            reason = "Stop Loss"

        elif row["Close"] >= target_price:
            reason = "Target"

        elif row["sell"]:
            reason = "UT Sell"

        else:
            continue

        exit_price = row["Close"]
        proceeds = pos["shares"] * exit_price
        profit = proceeds - pos["invested"]

        cash += proceeds

        trade_log.append({
            "Stock": ticker,
            "Entry Time": pos["entry_time"],
            "Exit Time": current_time,
            "Entry": pos["entry_price"],
            "Exit": exit_price,
            "Profit": profit,
            "Return %": (exit_price / pos["entry_price"] - 1) * 100,
            "Reason": reason
        })

        del open_positions[ticker]

    # ---------- BUY ----------
    if len(open_positions) < max_positions:

        for ticker, df in all_data.items():

            if ticker in open_positions:
                continue

            if current_time not in df.index:
                continue

            row = df.loc[current_time]

            if row["buy"]:

                allocation = cash / (max_positions - len(open_positions))

                if allocation <= 0:
                    break

                shares = allocation / row["Close"]

                cash -= allocation

                open_positions[ticker] = {
                    "entry_time": current_time,
                    "entry_price": row["Close"],
                    "shares": shares,
                    "invested": allocation
                }

    # ---------- EQUITY ----------
    total_value = cash

    for ticker, pos in open_positions.items():
        df = all_data[ticker]

        if current_time in df.index:
            total_value += pos["shares"] * df.loc[current_time]["Close"]

    equity_curve.append(total_value)

# ==========================
# RESULTS
# ==========================
trades = pd.DataFrame(trade_log)

print("\nTotal Trades:", len(trades))

if not trades.empty:
    win_rate = len(trades[trades["Profit"] > 0]) / len(trades)
    print("Win Rate:", win_rate)

final_capital = equity_curve[-1]

print("\nInitial Capital:", initial_capital)
print("Final Capital:", final_capital)
print("Return %:", (final_capital / initial_capital - 1) * 100)

# ==========================
# SAVE LOG
# ==========================
if not trades.empty:
    trades.to_csv("intraday_trades.csv", index=False)
    print("Saved to intraday_trades.csv")

# ==========================
# EQUITY CURVE
# ==========================
plt.figure()
plt.plot(equity_curve)
plt.title("Intraday UT Bot Equity Curve")
plt.xlabel("Time")
plt.ylabel("Capital")
plt.show()

# ==========================
# STATS
# ==========================
if not trades.empty:
    print("\nBest Trade:", trades["Profit"].max())
    print("Worst Trade:", trades["Profit"].min())